In [0]:
%pip install shap

1- Training models

In [0]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    cross_val_score
)

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

import matplotlib.pyplot as plt

import shap

2- Load data

In [0]:
df = pd.read_csv("hydrogen_feature_matrix.csv")

print(df.shape)

display(df.head())

3- Define x and y

In [0]:
TARGET = "log_hydrogen_production_rate"

X = df.drop(
    columns=[TARGET]
)

y = df[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)

numeric_features = X.select_dtypes(
    include="number"
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude="number"
).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", categorical_features)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_features
        ),
        (
            "categorical",
            make_pipeline(
                SimpleImputer(strategy="most_frequent"),
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            ),
            categorical_features
        )
    ]
)

4- Train/Test Split

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

5- Random Forest

In [0]:
rf = make_pipeline(
    preprocessor,
    RandomForestRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    )
)

rf.fit(
    X_train,
    y_train
)

pred_rf = rf.predict(X_test)

print("Random Forest")
print("R² :", r2_score(y_test, pred_rf))
print("MAE:", mean_absolute_error(y_test, pred_rf))
print(
    "RMSE:",
    np.sqrt(
        mean_squared_error(
            y_test,
            pred_rf
        )
    )
)

6- Gradient Boosting

In [0]:
gb = make_pipeline(
        preprocessor,
    GradientBoostingRegressor(
        random_state=42
    )
)

gb.fit(
    X_train,
    y_train
)

pred_gb = gb.predict(X_test)

print("Gradient Boosting")
print("R² :", r2_score(y_test, pred_gb))
print("MAE:", mean_absolute_error(y_test, pred_gb))
print(
    "RMSE:",
    np.sqrt(
        mean_squared_error(
            y_test,
            pred_gb
        )
    )
)

7- Cross Validation



In [0]:
scores = cross_val_score(
    rf,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("Random Forest CV")
print("Mean R²:", scores.mean())
print("Std R² :", scores.std())

8- Save predictions

In [0]:
predictions = pd.DataFrame({

    "experimental_log_h2": y_test,

    "predicted_log_h2": pred_gb

})

predictions.to_csv(
    "hydrogen_predictions.csv",
    index=False
)

predictions.head()

9- Parity plot

In [0]:
errors = pred_gb - y_test

plt.figure(figsize=(7,7))

sc = plt.scatter(
    y_test,
    pred_gb,
    c=errors,
    cmap="coolwarm",
    s=80,
    alpha=0.8
)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "--",
    linewidth=2,
    label="Perfect Prediction"
)

plt.text(
    0.05,
    0.95,
    (
        f"R² = {r2_score(y_test, pred_gb):.3f}\n"
        f"MAE = {mean_absolute_error(y_test, pred_gb):.3f}"
    ),
    transform=plt.gca().transAxes,
    verticalalignment="top",
    bbox=dict(
        boxstyle="round",
        facecolor="white",
        alpha=0.9
    )
)

cbar = plt.colorbar(sc)
cbar.set_label("Prediction Error")

plt.xlabel(
    "Experimental log(H₂ rate)"
)

plt.ylabel(
    "Predicted log(H₂ rate)"
)

plt.title(
    "Hydrogen Production Prediction Performance\n"
    "Red = Overprediction | Blue = Underprediction"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    "hydrogen_parity_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

10 - Feature importance

In [0]:
importance_df = pd.DataFrame({

    "feature": X.columns,

    "importance": gb.steps[-1][1].feature_importances_
})

importance_df = importance_df.sort_values(
    "importance",
    ascending=False
)

importance_df.to_csv(
    "hydrogen_feature_importance.csv",
    index=False
)

importance_df.head(20)

11 - Feature importance plot

In [0]:
top20 = importance_df.head(20)

import matplotlib.colors as mcolors

norm = mcolors.Normalize(
    vmin=top20["importance"].min(),
    vmax=top20["importance"].max()
)

colors = plt.cm.coolwarm(
    norm(top20["importance"])
)

plt.figure(figsize=(10,8))

bars = plt.barh(
    top20["feature"][::-1],
    top20["importance"][::-1],
    color=colors[::-1]
)

for bar in bars:

    width = bar.get_width()

    plt.text(
        width + 0.002,
        bar.get_y() + bar.get_height()/2,
        f"{width:.3f}",
        va="center",
        fontsize=9
    )

plt.xlabel(
    "Feature Importance"
)

plt.ylabel(
    "Feature"
)

plt.title(
    f"Top 20 Features for Hydrogen Prediction\n"
    f"Gradient Boosting (R² = {r2_score(y_test,pred_gb):.3f})"
)

plt.grid(
    axis="x",
    linestyle="--",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "hydrogen_feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

12- SHAP

In [0]:
gb_imputer = gb.steps[0][1]
gb_model = gb.steps[-1][1]

X_test_imputed = pd.DataFrame(
    gb_imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

explainer = shap.TreeExplainer(gb_model)

shap_values = explainer.shap_values(X_test_imputed)

13- SHAP plot

In [0]:
shap.summary_plot(
    shap_values,
    X_test,
    max_display=15,
    show=False
)

plt.tight_layout()

plt.savefig(
    "hydrogen_shap_summary.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

14 - Top feature table

In [0]:
top20 = importance_df.head(20)

display(top20)

15 - Back-transform to real H2 rates

In [0]:
real_units = pd.DataFrame({

    "experimental_h2":

        np.power(
            10,
            y_test
        ) - 1,

    "predicted_h2":

        np.power(
            10,
            pred_gb
        ) - 1

})

display(real_units.head())